# Gemma 4 quickstart

This notebook reuses the image/text example from [the model guide](../docs/models/gemma4.md). Run it in a dedicated environment with checkpoint access and Hugging Face authentication.

```bash
python -m pip install -e ".[gemma4,notebooks]"
```

Select that environment as the notebook kernel.


This example requires access to the checkpoint on Hugging Face and enough
GPU memory to load it. Accept the model's access terms and authenticate in
your environment before downloading its weights. Replace `example.jpg`
with your image. Native preprocessing expands the image placeholders;
pass all processor fields through to the probe.


In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, Gemma4ForConditionalGeneration
from vlmprobe import Prober

model_id = "google/gemma-4-E2B-it"
device, dtype = "cuda:0", torch.bfloat16
model = Gemma4ForConditionalGeneration.from_pretrained(
    model_id, dtype=dtype, device_map={"": device}, attn_implementation="eager",
).eval()
processor = AutoProcessor.from_pretrained(model_id)
probe = Prober(model, processor)

messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": "What is in this image?"},
]}]
prompt = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs = probe.prepare(
    prompt=prompt, image=Image.open("example.jpg").convert("RGB"), device=device,
)
for key, value in inputs.kwargs.items():
    if isinstance(value, torch.Tensor) and value.is_floating_point():
        inputs.kwargs[key] = value.to(dtype=dtype)

print(probe.describe()["methods"])
last = max(probe.spec.residuals)
result = probe.lens.logit(layers=[0, last], tokens="last_prompt").run(inputs)
top_id = result.tensors["logits"][-1, 0].argmax().item()
print(processor.tokenizer.decode([top_id]))
result.save("outputs/gemma4/lens")

profile = probe.attention.profile(layers=[last], queries="last_prompt").run(inputs)
print(profile.tensors["group_mass"])


For causal interventions, explicitly choose the answer metric. A single-token
margin measures the specified vocabulary contrast, not complete-answer
accuracy; use `SequenceLogProb` for a teacher-forced multi-token answer.


In [ ]:
from vlmprobe import TokenMargin

# Illustrative next-token contrast. Inspect the tokenizer's exact subwords.
positive = processor.tokenizer.encode(" cat", add_special_tokens=False)[0]
negative = processor.tokenizer.encode(" dog", add_special_tokens=False)[0]
effect = probe.causal.knockout(
    layers=[last], queries="last_prompt", keys="visual",
).run(inputs, metric=TokenMargin(positive, negative))
print(effect.tensors["baseline_score"], effect.tensors["effect"])


For text-only prompts on a multimodal checkpoint such as E2B, keep
`Gemma4ForConditionalGeneration` and omit the image; the automatic layout
contains no visual tokens. Do not load E2B with `Gemma4ForCausalLM`: its text
weights are stored under different names, so every weight is newly initialized
with only a load report and no exception. Use that class only for a checkpoint
whose weights match it, and check `output_loading_info=True`.
